# Frame B pre-build SMOKE — Stage 1 (variance decomposition, zero new code)

**DIAGNOSTIC — does NOT graduate anything.** This is the cheap first gate on the LEVEL→RATE reframe.
Per `brainstorm-workspace/2026-05-29-frameb-grill/04-smoke-harness-DRAFT.md`.

**Why Colab.** The structural per-seed variance the reframe hinges on only appears on real
wikitext at scale. STATUS.md (2026-05-28): *“local toy-corpus decomposition was inconclusive
(toy produces ~0 structural variance — can’t model real σ≈0.19) … the atom-vs-window microsplit
needs the real op point.”* So MPS/CPU local cannot answer this; CUDA on the real op point can.

**What it tests (grill items 3 / 6 / 7, the doc-unprovable premises):** Frame B assumes per-seed
codebook luck is *largely an additive intercept* that a within-seed slope differences out. This
decomposition measures whether per-seed luck is (a) structural at all, and (b) whether the two
worlds’ lifts move together — i.e. whether within-pair differencing actually cancels it.

**Reads → decision:**
- `dominant_source == "atom-draw"` AND `corr_AC_BD` low/negative → per-seed luck is structural
  and differencing does NOT cancel it → **the slope reframe likely buys little power**
  (item 3 fires at the endpoint level already). Consider NOT reframing before Stage 2.
- `corr_AC_BD` high (> ~0.3) → the worlds’ lifts co-move, differencing cancels seed variance
  → reframe is promising → proceed to Stage 2 (checkpoint hook) to confirm at the slope level.
- `dominant_source == "window-draw"` → cheaper fix is averaging window draws, not the slope.
- `dominant_source == "binomial"` → op point too small to see structure; raise n_test and re-run.

**Holdout seeds (item-6 contamination fix):** atom seeds **1000..1009**, disjoint from the
graduation seeds 0..9, so the σ used to justify the reframe is never read on the verdict seeds.

**Runtime.** `--variance-decomp` runs atom_seeds × window_seeds × 2 cells (A consolidated, C frozen)
at the real wikitext op point. 10 atom × 4 window × 2 = 80 cells ≈ the Gate-0 full-run scale
(~30–75 min on A100/L4). Deterministic — if the runtime drops, re-run.

**Pre-flight.** Assumes branch `codex/phase5-prime-bundle-first-scene-memory` is pushed (it is:
the variance tools are commit `43cbb1b`).

In [ ]:
# 1. Clone the repo + verify the variance tools are present on the branch.
import os
from pathlib import Path
REPO_DIR = '/content/Neuro-AI'
BRANCH = 'codex/phase5-prime-bundle-first-scene-memory'
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout {BRANCH}
!git log --oneline -3
os.chdir(REPO_DIR)

gate0 = Path(REPO_DIR) / 'experiments' / 'gate0_frame_a.py'
driver = Path(REPO_DIR) / 'experiments' / 'c3_phase3_exit_criterion.py'
if not gate0.exists():
    raise SystemExit('experiments/gate0_frame_a.py not found — push the branch and re-run.')
gsrc = gate0.read_text()
dsrc = driver.read_text()
for needle, where in [
    ('def run_variance_decomposition', 'gate0 variance-decomposition tool'),
    ('def variance_report_from_summary', 'gate0 variance-report tool'),
    ('--variance-decomp', 'gate0 --variance-decomp CLI'),
]:
    if needle not in gsrc:
        raise SystemExit(f'{where} missing — push the branch and re-run.')
if 'window_seed_override' not in dsrc:
    raise SystemExit('c3 window_seed_override missing — push the branch and re-run.')
print('Variance tools verified on branch.')

In [ ]:
# 2. Mount Drive for result persistence (and optional Stage-1B re-analysis input).
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
print('Drive mounted.')

In [ ]:
# 3. Install deps.
!pip install -q torch datasets huggingface_hub

In [ ]:
# 4. Pre-warm the WikiText-2 HF cache (parent CPU-only; the run below is single CUDA process).
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from datasets import load_dataset
_ = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train[:1%]')
print('WikiText-2 cache pre-warmed.')

In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv

In [ ]:
# 6. SMOKE — tiny synthetic variance-decomp on CUDA to confirm runtime + plumbing.
#    Synthetic corpus has ~0 structural variance, so expect dominant_source='binomial'
#    (or window-draw) — this is a plumbing check, NOT a real read.
import subprocess, sys, os, json
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/frameb_smoke_stage1_synthsmoke')
smoke_out.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'experiments/gate0_frame_a.py', '--variance-decomp',
       '--atom-seeds', '1000,1001', '--window-seeds', '0,1',
       '--corpus-source', 'synthetic', '--vocab-size', '40',
       '--D', '256', '--landscape-size', '8', '--window', '4',
       '--n-test-windows', '24', '--n-train-windows', '48',
       '--n-consolidation-events', '50',
       '--device', 'cuda', '--output-dir', str(smoke_out)]
rc = subprocess.call(cmd)
print(f'smoke exit code: {rc}')
if rc != 0:
    raise SystemExit('Stage-1 smoke failed — abort before the full run.')
print('smoke OK — runtime + variance-decomp plumbing confirmed.')

In [ ]:
# 7. STAGE 1A — nested atom×window variance decomposition at the Path C wikitext op point.
#    HOLDOUT atom seeds 1000..1009 (item-6 fix). 4 window seeds per atom.
#    Op point MUST match Gate 0 (D=4096, landscape=64, window=8, n_test=512,
#    n_train=2048, beta=10, n_consolidation_events=1000) so σ is comparable to 0.196.
import subprocess, sys, os
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
out_dir = Path('reports/frameb_smoke_stage1_2026-05-29')
out_dir.mkdir(parents=True, exist_ok=True)
log = out_dir / 'run.log'
cmd = [sys.executable, 'experiments/gate0_frame_a.py', '--variance-decomp',
       '--atom-seeds', '1000,1001,1002,1003,1004,1005,1006,1007,1008,1009',
       '--window-seeds', '0,1,2,3',
       '--D', '4096', '--landscape-size', '64', '--window', '8',
       '--n-test-windows', '512', '--n-train-windows', '2048',
       '--K', '5', '--beta', '10', '--n-consolidation-events', '1000',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--lr-pull', '0.1', '--lr-push', '0.05',
       '--corpus-source', 'wikitext', '--wikitext-name', 'wikitext-2-raw-v1',
       '--vocab-cap', '1000',
       '--device', 'cuda', '--output-dir', str(out_dir)]
with log.open('w') as f:
    rc = subprocess.call(cmd, stdout=f, stderr=subprocess.STDOUT)
print(f'variance-decomp exit code: {rc}')
!tail -40 {log}
if rc != 0:
    raise SystemExit('Stage-1A full run failed — see log above.')

In [ ]:
# 8. Copy results to Drive.
import shutil, os
dst = '/content/drive/MyDrive/neuro-ai/results/frameb_smoke_stage1_2026-05-29'
os.makedirs(dst, exist_ok=True)
shutil.copytree('reports/frameb_smoke_stage1_2026-05-29', dst, dirs_exist_ok=True)
print('copied to', dst)
!ls -la {dst}

In [ ]:
# 9. STAGE 1A READ — surface the decomposition + the reframe decision.
import json
rep = json.load(open('reports/frameb_smoke_stage1_2026-05-29/variance_decomp.json'))
print('dominant_source :', rep['dominant_source'])
print('var_fraction_atom:', rep['var_fraction_atom'])
print('sd_total         :', round(rep['sd_total'], 4), ' (compare to endpoint σ≈0.196)')
print('sd_between_atom  :', round(rep['sd_between_atom'], 4))
print('sd_window_draw   :', round(rep['sd_window_draw_binom_subtracted'], 4))
print('sd_binomial_floor:', round(rep['sd_binomial_floor_estimate'], 4))
print('\nvariance_components:', json.dumps(rep['variance_components'], indent=2))
print('\nrecommendation:\n', rep['recommendation'])
print('\n--- REFRAME READ (manual) ---')
print('atom-draw dominant + low corr  => slope reframe buys little power (reconsider).')
print('high corr / co-moving lifts    => reframe promising => proceed to Stage 2.')
print('NOTE: this decomposes the ENDPOINT lift A−C. The slope-specific gates')
print('(corr(intercept,slope), fanning, E[β_shuffle]) still require the Stage-2 checkpoint hook.')

In [ ]:
# 10. STAGE 1B (OPTIONAL, zero compute) — decompose the EXISTING Gate-0 run's DiD variance.
#     Only runs if you still have the n=10 Gate-0 summary on Drive. No re-run; pure re-analysis.
import os, subprocess, sys, json
candidates = [
    '/content/drive/MyDrive/neuro-ai/results/gate0_2026-05-28/gate0_summary.json',
    'reports/gate0_2026-05-28/gate0_summary.json',
]
src = next((p for p in candidates if os.path.exists(p)), None)
if src is None:
    print('No Gate-0 summary found on Drive or locally — skipping Stage 1B.')
    print('(Stage 1A above is self-contained and sufficient.)')
else:
    os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
    out = subprocess.check_output(
        [sys.executable, 'experiments/gate0_frame_a.py',
         '--variance-report', src]).decode()
    print('Stage 1B — DiD variance decomposition of the existing Gate-0 run:\n')
    print(out)
    rep = json.loads(out)
    print('\nKEY READS:')
    print('  corr_AC_BD                :', rep.get('corr_AC_BD'),
          '(>~0.3 => differencing cancels seed variance => reframe promising)')
    print('  did_sd_over_binomial_floor:', rep.get('did_sd_over_binomial_floor'),
          '(>>1 => structural, not sampling noise)')
    print('  n_for_80pct_power_at_effect:', rep.get('n_for_80pct_power_at_effect'))